# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [3]:
# Đọc dữ liệu từ bảng Lop trong CSDL libol
query_Vatmangtin = "SELECT Vat_mang_tin_ID, dbo.DecodeUTF8String(Ky_hieu) AS Ky_hieu, dbo.DecodeUTF8String(Vat_mang_tin) AS Vat_mang_tin FROM Vat_mang_tin"
df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_libol)
print(df_vatmangtin)

    Vat_mang_tin_ID Ky_hieu          Vat_mang_tin
0                 1      MC             Microfilm
1                 2      MF            Microfiche
2                 3       G                  Giấy
3                 4      BT               Băng từ
4                 5      ĐT                Đĩa từ
5                 6      CD  Đĩa CDROM (đĩa Laze)
6                 7      VA    Vật liệu nghe nhìn
7                 8    giấy                  None
8                 9  Dia tu                  None
9                10      GH                  None
10               11       B                  None
11               12     Tan                  None
12               13   Ebook          Sách điện tử
13               14      VT                  None
14               15       A                  None


C:\Users\admin\AppData\Local\Temp\ipykernel_21612\2402062624.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_vatmangtin = pd.read_sql(query_Vatmangtin, conn_libol)


## Xử lý data

In [4]:
new_row = pd.DataFrame({'Vat_mang_tin_ID': [0],'Ky_hieu': ['None'], 'Vat_mang_tin': ['(Không xác định)']}) # Tạo hàng dữ liệu giả lập cho nhóm không xác định
df_vatmangtin = pd.concat([df_vatmangtin, new_row], ignore_index=True) # Thêm vào dataframe
df_vatmangtin = df_vatmangtin.sort_values(by="Vat_mang_tin_ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn
print(df_vatmangtin)

    Vat_mang_tin_ID Ky_hieu          Vat_mang_tin
0                 0    None      (Không xác định)
1                 1      MC             Microfilm
2                 2      MF            Microfiche
3                 3       G                  Giấy
4                 4      BT               Băng từ
5                 5      ĐT                Đĩa từ
6                 6      CD  Đĩa CDROM (đĩa Laze)
7                 7      VA    Vật liệu nghe nhìn
8                 8    giấy                  None
9                 9  Dia tu                  None
10               10      GH                  None
11               11       B                  None
12               12     Tan                  None
13               13   Ebook          Sách điện tử
14               14      VT                  None
15               15       A                  None


## Load data

### [Nếu cần] Clear bảng

In [5]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Vat_mang_tin"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [6]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Vat_mang_tin (ID_vat_mang_tin, Ky_hieu, Vat_mang_tin) 
                VALUES (?, ?, ?)
                """
for index, row in df_vatmangtin.iterrows():
    values = (row['Vat_mang_tin_ID'], 
              row['Ky_hieu'],
              row['Vat_mang_tin'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()